In [1]:
import pandas as pd

In [2]:
from openai import OpenAI
import os
from dotenv import load_dotenv

# Load .envrc (or .env) file
load_dotenv('../.envrc')
client = OpenAI()

In [3]:
df = pd.read_csv('../data/feature_store_data.csv')
documents = df.to_dict(orient="records")

In [4]:
documents[0]

{'id': 0,
 'feature_name': 'purchase_count_7d_web',
 'feature_group': 'customer_behavior',
 'computation_logic': "COUNT(order_id) WHERE order_status='DELIVERED' OVER last 7 days BY customer_id",
 'data_source': 'fct_orders (Silver)',
 'update_frequency': 'Hourly',
 'serving_store': 'DynamoDB, S3',
 'models_using_feature': 'recommendation_model, churn_model',
 'feature_description': 'Short-term weekly purchase count metric. Useful for RFM segmentation, customer lifetime value modeling, and personalized promotion campaigns. Captured from the web channel, representing customer interactions on the Amazon website.'}

In [5]:
prompt_template = """
You are a data scientist or machine learning engineer exploring the feature store.
Your task is to formulate 5 natural, business-focused questions that a user might ask
based on the provided feature.

Requirements:
- **2 questions** must be based on the `feature_description` (business context, purpose, use case)
- **3 questions** can be based on other fields (feature_name, feature_group, computation_logic, data_source, update_frequency, serving_store, models_using_feature)

Make the questions:
- Complete and conversational (not too short)
- Specific to this feature's business purpose
- Framed around real-world use cases

The record:

feature_name: {feature_name}
feature_group: {feature_group}
computation_logic: {computation_logic}
data_source: {data_source}
update_frequency: {update_frequency}
serving_store: {serving_store}
models_using_feature: {models_using_feature}
feature_description: {feature_description}

Example questions for a feature like "purchase_count_7d":
Description-based questions (2):
- "How can I track a customer's recent purchase activity to identify active buyers?"
- "Which metric should I use to segment customers for a weekly loyalty campaign?"

Other questions (3):
- "What feature name measures a customer's 7-day purchase behavior?"
- "How often is the purchase_count_7d feature updated?"
- "Which models use the 7-day purchase count feature?"

Provide the output in parsable JSON without using code blocks:

{{"questions": ["question1", "question2", "question3", "question4", "question5"]}}
""".strip()

In [6]:
prompt = prompt_template.format(**documents[0])

In [7]:
def llm(prompt):
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

In [8]:
questions = llm(prompt)

In [9]:
import json

In [10]:
json.loads(questions)

{'questions': ['How can the purchase_count_7d_web feature help us identify potential customers for targeted marketing strategies?',
  'In what ways can we leverage the 7-day purchase count to enhance our customer retention initiatives?',
  'What customer behavior does the purchase_count_7d_web feature specifically quantify?',
  'How frequently is the purchase_count_7d_web updated, and how does this impact our real-time marketing efforts?',
  'Which machine learning models are utilizing the purchase_count_7d_web feature to drive predictions, and how can we improve their accuracy?']}

In [11]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)

    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{"role": "user", "content": prompt}]
    )

    json_response = response.choices[0].message.content
    return json_response

In [12]:
from tqdm.auto import tqdm

In [13]:
results = {}

In [14]:
for doc in tqdm(documents): 
    doc_id = doc['id']
    if doc_id in results:
        continue

    questions_raw = generate_questions(doc)
    questions = json.loads(questions_raw)
    results[doc_id] = questions['questions']

  0%|          | 0/72 [00:00<?, ?it/s]

In [15]:
results[1]

['How can understanding the bi-weekly purchase count help us in tailoring our marketing strategies for repeat customers?',
 'In what ways can we utilize the purchase_count_14d_web feature to effectively segment customers for personalized promotion campaigns?',
 "What is the significance of the feature name 'purchase_count_14d_web' in analyzing customer behavior over the last two weeks?",
 'How frequently is the purchase_count_14d_web feature updated, and how does that frequency impact our real-time marketing efforts?',
 'Could you explain how the recommendation_model and churn_model leverage the purchase_count_14d_web feature to enhance customer engagement and retention?']

In [16]:
final_results = []

for doc_id, questions in results.items():
    for q in questions:
        final_results.append((doc_id, q))

In [17]:
final_results[0]

(0,
 'How can the purchase_count_7d_web feature help in determining the right time to launch targeted promotions for our customers?')

In [18]:
df_results = pd.DataFrame(final_results, columns=['id', 'question'])

In [19]:
df_results.to_csv('../data/ground-truth-retrieval.csv', index=False)

In [20]:
!head ../data/ground-truth-retrieval.csv

id,question
0,How can the purchase_count_7d_web feature help in determining the right time to launch targeted promotions for our customers?
0,In what ways can this short-term purchase count metric be leveraged for accurately estimating customer lifetime value in our business model?
0,"What is the frequency of updates for the purchase_count_7d_web feature, and how does it affect our marketing strategies?"
0,Can you tell me more about the data source for the purchase_count_7d_web feature and how reliable it is in measuring customer behavior?
0,Which models are currently utilizing the purchase_count_7d_web feature to enhance their predictions or recommendations for customers?
1,How can understanding the bi-weekly purchase count help us in tailoring our marketing strategies for repeat customers?
1,In what ways can we utilize the purchase_count_14d_web feature to effectively segment customers for personalized promotion campaigns?
1,What is the significance of the feature name 'purchase_coun